``ReliabilityModel().fit()`` records everything needed to judge how much to trust a
prediction: the training feature distribution (applicability domain), the uncertainty
source, an optional probability calibrator, and the split-conformal calibration.

In [1]:
import aaanalysis as aa
import numpy as np
from sklearn.datasets import make_classification
aa.options["verbose"] = False
# ``X`` is any feature matrix — e.g. a CPP feature matrix from
# ``SequenceFeature.feature_matrix`` — and ``labels`` are binary. A compact synthetic
# stand-in is used here so the example runs in a second.
X, labels = make_classification(n_samples=140, n_features=10, n_informative=6, random_state=42)
X_train, labels_train, X_new = X[:110], labels[:110], X[110:]

With ``model=None`` a default random forest is fitted and its bootstrap disagreement
becomes the uncertainty; ``verbose`` and ``random_state`` are set on the constructor.

In [2]:
rm = aa.ReliabilityModel(random_state=42, verbose=False).fit(X=X_train, labels=labels_train)

Pass a **list** of fitted estimators (or a fitted :class:`~aaanalysis.AAPred`) as
``model`` to use their disagreement as the uncertainty instead of the bootstrap.

In [3]:
from sklearn.ensemble import RandomForestClassifier
models = [RandomForestClassifier(n_estimators=50, random_state=i).fit(X_train, labels_train)
          for i in range(4)]
rm = aa.ReliabilityModel(random_state=42).fit(X=X_train, labels=labels_train, model=models)

**Further parameters.** ``label_pos`` is the positive class; ``k`` and ``ad_percentile``
shape the applicability-domain boundary; ``ci`` is the reported interval width;
``n_bootstrap`` sets the resamples for a single-model uncertainty; ``calibrate`` /
``calibration_method`` fit the probability calibrator; ``conformal_alpha`` is the
conformal miscoverage level.

In [4]:
rm = aa.ReliabilityModel(random_state=42).fit(
    X=X_train, labels=labels_train, model=None, label_pos=1, k=5, ad_percentile=95,
    ci=90, n_bootstrap=20, calibrate=True, calibration_method="isotonic",
    conformal_alpha=0.1)